# Mamba-SSM Oracle — single diagnostic comparison

**Project: `transformer-vs-ssm` (sequence-modeling blog series). Unrelated to any competition.**

## Why this exists

After fixing three real bugs in the from-scratch Mamba (`dt_proj` init, a blanket
reinit that erased it, and the missing `dt_rank` bottleneck), plus proving scan
parity against the official reference (`tests/test_selective_scan_parity.py`,
passes), the corrected model **still fails Selective Copying** at `ssm_lr_scale=1`
(commit `58ebc92`) -- flat loss at ln(64) through 800 steps, val_acc 0.008-0.012
vs. chance 0.032.

Per external review: when scan parity passes and the model still fails, the next
discriminating experiment is the **official `mamba-ssm` package as an oracle** --
same task, same dimensions, same optimizer/schedule -- to find out whether the
remaining gap is (a) still something specific to our implementation, or (b) the
task/training recipe itself at this scale/step budget.

## Decision table

| Custom Mamba | Official `mamba-ssm` | Interpretation |
|---|---|---|
| fails | **learns** | model-level discrepancy remains (wrapper/precision/training path) |
| fails | **fails** | stop debugging the architecture -- investigate task/recipe first |
| learns | learns | custom implementation validated |
| learns | fails | harness/config mismatch -- don't interpret either yet |

This notebook trains **three models, one seed each, same task/steps**, for a clean
comparison in one environment: Transformer (known-good control), our custom Mamba
(post-fix), official `mamba_ssm.modules.mamba_simple.Mamba` (Mamba-1, matching
what we've been debugging -- not Mamba-2).

## Kaggle settings
- Accelerator: **GPU T4 x1**, Internet **ON**
- Save Version -> Run All
- If `mamba-ssm` fails to install/import, the notebook stops with a clear
  diagnostic rather than failing opaquely later.

In [ ]:
# Cell 1: Clone (or refresh) the repo -- picks up the ablation fix (7ec4346)
import os, subprocess, sys, platform

REPO = "https://github.com/nvaidyan1/transformer-vs-ssm.git"
if not os.path.exists("transformer-vs-ssm"):
    subprocess.run(["git", "clone", REPO], check=True)

os.chdir("/kaggle/working/transformer-vs-ssm")
subprocess.run(["git", "pull", "--ff-only"], check=False)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

import torch
print(f"\npython {platform.python_version()}  torch {torch.__version__}  "
      f"cuda {torch.version.cuda}  "
      f"gpu {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

In [ ]:
# Cell 2: Repo dependencies + full test suite (must pass before spending GPU time)
!pip install -q pyyaml pytest einops 2>&1 | tail -1
!python -m pytest tests/ -q 2>&1 | tail -6

In [ ]:
# Cell 3: Install mamba-ssm. causal-conv1d is optional (best-effort acceleration
# for the depthwise conv; the module falls back to plain PyTorch conv without it).
# mamba-ssm itself is NOT optional -- its compiled selective_scan extension is
# imported unconditionally by mamba_simple.py, so a failed build here blocks the
# whole notebook.
#
# Attempt 1 (kernel v1) failed at "Getting requirements to build wheel" -- pip's
# outer error, with the real cause hidden because `-q` suppressed the subprocess's
# own output. Root cause (well-documented for this package): mamba-ssm's setup.py
# needs to introspect the ALREADY-INSTALLED torch/CUDA version to pick the right
# build path, but pip's isolated build sandbox (PEP 517) doesn't have torch
# visible by default. Fix: --no-build-isolation, plus pre-installing the build-time
# deps setup.py itself needs (packaging, ninja, wheel, psutil). No -q this time --
# if it fails again, the full output below must be enough to diagnose it further,
# not just "exit code 1" again.
import subprocess

def run(cmd):
    print(f"$ {' '.join(cmd)}")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
    print(f"[exit {r.returncode}]\n")
    return r

run(["pip", "install", "packaging", "ninja", "wheel", "psutil"])

print("Attempting causal-conv1d (optional, --no-build-isolation)...")
r = run(["pip", "install", "causal-conv1d>=1.4.0", "--no-build-isolation"])
print("  OK" if r.returncode == 0 else "  not available (non-fatal) -- see output above")

print("\nAttempting mamba-ssm (required, --no-build-isolation)...")
r = run(["pip", "install", "mamba-ssm", "--no-build-isolation"])
if r.returncode != 0:
    print("=" * 70)
    print("mamba-ssm FAILED TO INSTALL even with --no-build-isolation. Full output")
    print("is above (not truncated this time). The oracle experiment cannot")
    print("proceed on this environment -- stop here rather than fail later.")
    print("=" * 70)
    raise SystemExit("mamba-ssm install failed -- see diagnostic above")
print("  OK")

try:
    from mamba_ssm.modules.mamba_simple import Mamba as OfficialMamba
    import mamba_ssm
    print(f"\nmamba_ssm {mamba_ssm.__version__} imported OK")
except Exception as e:
    print("=" * 70)
    print(f"mamba-ssm installed but failed to IMPORT: {type(e).__name__}: {e}")
    print("Likely a torch/cuda ABI mismatch between the installed wheel and this")
    print("environment. Stop here -- see the version print in Cell 1.")
    print("=" * 70)
    raise

In [ ]:
# Cell 4: Oracle wrapper -- residual pre-norm block around the official Mamba-1
# mixer, matching our own MambaBlock structure (x + mixer(norm(x))) and our
# model's forward(x) -> logits interface, so train/eval code can be shared.
#
# use_fast_path is exposed and passed straight through to the official Mamba
# constructor. This is the discriminator recommended after the first oracle
# result: with use_fast_path=True it calls the fused mamba_inner_fn; with it
# False, it separately runs conv1d, x_proj, dt_proj, selective_scan_fn, then
# out_proj -- same parameters and architecture, different execution path.
# Note use_fast_path=False can still use optimized causal_conv1d_fn/
# selective_scan_fn CUDA ops if installed -- it is not "pure PyTorch", so if
# it also learns well, the next narrowing step is module-level weight-copy
# parity, not "CUDA vs Python".
import torch
import torch.nn as nn

class OracleBlock(nn.Module):
    def __init__(self, d_model, d_state, d_conv, expand, layer_idx, use_fast_path=True):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mixer = OfficialMamba(d_model=d_model, d_state=d_state, d_conv=d_conv,
                                   expand=expand, layer_idx=layer_idx,
                                   use_fast_path=use_fast_path)

    def forward(self, x):
        return x + self.mixer(self.norm(x))

class OracleMamba(nn.Module):
    def __init__(self, vocab_size, n_layers, d_model, d_state, d_conv, expand,
                use_fast_path=True):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            OracleBlock(d_model, d_state, d_conv, expand, layer_idx=i,
                       use_fast_path=use_fast_path)
            for i in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying, matching our model

    def forward(self, x):
        h = self.tok_emb(x)
        for block in self.blocks:
            h = block(h)
        return self.head(self.ln_f(h))

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
# Cell 5: Shared settings -- identical across all three models
TASK_ARGS = dict(n_data=4, seq_len=64, vocab_size=64)
D_MODEL, D_STATE, D_CONV, EXPAND, N_LAYERS = 124, 16, 4, 2, 4  # matches mamba_small.yaml
BATCH, LR, WARMUP, STEPS, SEED = 32, 3e-4, 200, 2000, 0
OUT = "/kaggle/working/oracle"

import os
os.makedirs(OUT, exist_ok=True)
print(f"task={TASK_ARGS}  d_model={D_MODEL} d_state={D_STATE} d_conv={D_CONV} "
      f"expand={EXPAND} n_layers={N_LAYERS}\n"
      f"batch={BATCH} lr={LR} warmup={WARMUP} steps={STEPS} seed={SEED}")

In [ ]:
# Cell 6: Transformer -- known-good control, via the existing harness (unchanged)
task_args_str = ",".join(f"{k}={v}" for k, v in TASK_ARGS.items())
!python src/train_task.py --config configs/recall/transformer_small.yaml \
    --task selective_copy --task_args {task_args_str} --steps {STEPS} \
    --batch_size {BATCH} --seed {SEED} --device cuda \
    --out {OUT}/transformer.json 2>&1 | tail -6

In [ ]:
# Cell 7: Our custom Mamba, post-fix, ssm_lr_scale=1 (default -- no boost)
!python src/train_task.py --config configs/recall/mamba_small.yaml \
    --task selective_copy --task_args {task_args_str} --steps {STEPS} \
    --batch_size {BATCH} --lr {LR} --seed {SEED} --device cuda \
    --out {OUT}/custom_mamba.json 2>&1 | tail -6

In [ ]:
# Cell 8: Official mamba-ssm oracle, trained TWICE -- use_fast_path=True and
# =False -- same seed, same everything else. Discriminator recommended after
# the first oracle result: if both learn similarly, fusion is exonerated and
# the gap is at the block/model level, not CUDA-fused-vs-naive execution. If
# only the fast path learns, the execution path itself is implicated.
import sys, math, time, json
sys.path.insert(0, "/kaggle/working/transformer-vs-ssm")
import torch
import torch.nn as nn
from src.tasks import IGNORE_INDEX, make_batcher

def train_oracle(use_fast_path, out_path, seed=SEED, steps=STEPS):
    torch.manual_seed(seed)
    oracle = OracleMamba(vocab_size=TASK_ARGS["vocab_size"], n_layers=N_LAYERS,
                         d_model=D_MODEL, d_state=D_STATE, d_conv=D_CONV,
                         expand=EXPAND, use_fast_path=use_fast_path).to(device)
    n_params = count_params(oracle)

    decay, no_decay = [], []
    for n, p in oracle.named_parameters():
        (no_decay if getattr(p, "_no_weight_decay", False) else decay).append(p)
    opt = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 0.1}, {"params": no_decay, "weight_decay": 0.0}],
        lr=LR, betas=(0.9, 0.95))

    def get_lr(step, warmup=WARMUP, max_steps=steps, lr=LR):
        if step < warmup:
            return lr * (step + 1) / warmup
        prog = (step - warmup) / max(1, max_steps - warmup)
        return lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))

    batcher = make_batcher("selective_copy", **TASK_ARGS)
    t0 = time.time()
    oracle.train()
    history = []
    for step in range(steps):
        lr = get_lr(step)
        for g in opt.param_groups:
            g["lr"] = lr
        tok, tgt, _, _ = batcher(BATCH, seed=step + seed * 10_000_000)
        tok = torch.from_numpy(tok).to(device)
        tgt = torch.from_numpy(tgt).to(device)
        logits = oracle(tok)
        loss = nn.functional.cross_entropy(logits.reshape(-1, TASK_ARGS["vocab_size"]),
                                           tgt.reshape(-1), ignore_index=IGNORE_INDEX)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(oracle.parameters(), 1.0)
        opt.step()

        if (step + 1) % 250 == 0 or step == steps - 1:
            oracle.eval()
            with torch.no_grad():
                vlogits = oracle(val_tok)
                vloss = nn.functional.cross_entropy(
                    vlogits.reshape(-1, TASK_ARGS["vocab_size"]), val_tgt.reshape(-1),
                    ignore_index=IGNORE_INDEX)
                vacc = (vlogits.argmax(-1)[val_mask] == val_tgt[val_mask]).float().mean().item()
            oracle.train()
            history.append({"step": step + 1, "val_acc": vacc, "val_loss": vloss.item()})
            print(f"  [{'fast' if use_fast_path else 'slow'}] step {step+1:>5d}  "
                  f"train_loss {loss.item():.4f}  val_loss {vloss.item():.4f}  "
                  f"val_acc {vacc:.4f}  ({time.time()-t0:.0f}s)", flush=True)

    record = {"arch": f"mamba_ssm_oracle_{'fast' if use_fast_path else 'slow'}",
             "use_fast_path": use_fast_path, "n_params": n_params,
             "final_val_acc": vacc, "final_val_loss": vloss.item(),
             "steps_run": steps, "seed": seed,
             "wall_clock_s": round(time.time() - t0, 1),
             "chance_acc": 1.0 / ((TASK_ARGS["vocab_size"] - 1) // 2),
             "history": history}
    json.dump(record, open(out_path, "w"), indent=2)
    print(f"  final val_acc {vacc:.4f}  ({record['wall_clock_s']}s)\n")
    return record

device = torch.device("cuda")
val_tok, val_tgt, val_mask, _ = make_batcher("selective_copy", **TASK_ARGS)(64, seed=10_000_000)
val_tok, val_tgt, val_mask = (torch.from_numpy(val_tok).to(device),
                              torch.from_numpy(val_tgt).to(device),
                              torch.from_numpy(val_mask).to(device))

print("=== Official mamba-ssm, use_fast_path=True (fused) ===")
rec_fast = train_oracle(True, f"{OUT}/oracle_fast.json")

print("=== Official mamba-ssm, use_fast_path=False (unfused, still may use CUDA ops) ===")
rec_slow = train_oracle(False, f"{OUT}/oracle_slow.json")

In [ ]:
# Cell 9: Verdict -- trajectory-aware, not a single endpoint threshold.
#
# A hard endpoint cutoff (e.g. "val_acc > 0.5") mis-classifies a model that is
# clearly still improving at the step budget's end as a "fail", which is what
# happened in kernel v2 (oracle 0.46 and rising steeply was called the same as
# custom 0.13). The categories below look at both level and trend:
#
#   LEARNS       clearly above chance AND still improving meaningfully over the
#                second half of training (or already high and stable)
#   FLAT/FAIL    stays near chance, no meaningful improvement over training
#   INCONCLUSIVE improving, but not clearly separated from chance/the other run
import glob, json

results = {}
for f in glob.glob(f"{OUT}/*.json"):
    r = json.load(open(f))
    key = r.get("arch", r.get("task", "unknown"))
    results[key] = r

chance = 1.0 / ((TASK_ARGS["vocab_size"] - 1) // 2)

def classify(rec, chance):
    if rec is None:
        return "N/A", None
    acc = rec["final_val_acc"]
    hist = rec.get("history", [])
    if len(hist) >= 2:
        mid = hist[len(hist) // 2]["val_acc"]
        still_rising = (acc - mid) > 0.03  # meaningful improvement in the second half
    else:
        still_rising = False
    if acc < chance * 2:
        return "FLAT/FAIL", acc
    if acc > 0.5 or still_rising:
        return "LEARNS", acc
    return "INCONCLUSIVE", acc

rows = [
    ("transformer (control)", results.get("transformer")),
    ("custom mamba (post-fix)", results.get("mamba")),
    ("mamba-ssm oracle, fast", results.get("mamba_ssm_oracle_fast")),
    ("mamba-ssm oracle, slow", results.get("mamba_ssm_oracle_slow")),
]

print(f"{'run':26s} {'val_acc':>9s}  {'verdict':>12s}   (chance {chance:.3f})")
print("-" * 62)
for name, rec in rows:
    label, acc = classify(rec, chance)
    acc_str = f"{acc:.4f}" if acc is not None else "  n/a"
    print(f"{name:26s} {acc_str:>9s}  {label:>12s}")

cus_label, _ = classify(results.get("mamba"), chance)
fast_label, _ = classify(results.get("mamba_ssm_oracle_fast"), chance)
slow_label, _ = classify(results.get("mamba_ssm_oracle_slow"), chance)
tra_label, _ = classify(results.get("transformer"), chance)

print()
if tra_label != "LEARNS":
    print("INCONCLUSIVE -- even the transformer control did not clearly learn. "
          "Fix the harness before drawing any conclusion about either Mamba.")
elif fast_label == "LEARNS" and slow_label == "LEARNS" and cus_label != "LEARNS":
    print("Official LEARNS on both paths, custom does not -- fusion is exonerated. "
          "A model-level discrepancy remains (init at construction, residual/norm "
          "wrapper, optimizer grouping, or a training-level detail). Next: "
          "module-level forward/backward parity with weight-copying from the "
          "official block into ours, stage by stage (in_proj -> conv+SiLU -> "
          "x_proj -> dt_proj -> scan [already cleared] -> D+gate -> out_proj).")
elif fast_label == "LEARNS" and slow_label != "LEARNS":
    print("Official LEARNS fast, FAILS slow -- the fused execution path itself is "
          "implicated (not just faster, but different at the training-dynamics "
          "level). This is the more surprising and more interesting outcome -- "
          "worth confirming with a second seed before trusting it.")
elif fast_label != "LEARNS" and slow_label != "LEARNS":
    print("Neither official path clearly learns -- reconsider the oracle setup "
          "itself (dimension mismatch, an oracle-side bug) before concluding "
          "anything about the custom implementation.")
else:
    print("Mixed result outside the anticipated cases -- inspect the per-checkpoint "
          "histories in the saved JSON before interpreting.")

## Module-level parity (weight-copying)

v3 result: official `mamba-ssm` gives **bit-identical** trajectories on `use_fast_path=True`
and `=False` (same val_acc and val_loss at every checkpoint, e.g. 0.4609 / 1.9767 at step
2000 for both). Fusion is exonerated -- but checking the reference source
(`mamba_simple.py`) shows why: even the "slow" path still calls the opaque
`causal_conv1d_fn` / `selective_scan_fn` CUDA kernels when installed, so neither run was
actually "plain PyTorch". Both paths share the same conv/scan kernels; only the outer
projection fusion (`mamba_inner_fn`) differs, and that evidently doesn't change the
numbers.

So the next test bypasses `forward()` entirely on both sides: construct an official
`Mamba` layer and our own `MambaSSM` at matching dimensions, **copy the official
module's weights directly into ours** (parameter-for-parameter, shape-asserted first),
then compare forward output and gradients stage by stage on one batch, using each
model's own submodules manually -- not their `forward()` methods, so there is no
opaque kernel standing between us and the actual numbers being compared.

If everything downstream of "same weights, same input" matches, the architecture is
exonerated too, and the training-time gap comes from something else: initialization
*before* this test's weight-copying, the residual/norm wrapper, optimizer grouping, or a
training-level detail -- not the computation itself.

In [ ]:
# Cell 11: Weight-copy parity -- official Mamba vs our MambaSSM, stage by stage.
# Bypasses forward() on both sides; walks each model's own submodules manually so
# there is no fused/opaque kernel between us and the numbers.
import sys, math
sys.path.insert(0, "/kaggle/working/transformer-vs-ssm")
import torch
import torch.nn as nn
import torch.nn.functional as F
from mamba_ssm.modules.mamba_simple import Mamba as OfficialMamba
from src.models.mamba import MambaSSM, selective_scan

torch.manual_seed(0)
device = torch.device("cuda")

official = OfficialMamba(d_model=D_MODEL, d_state=D_STATE, d_conv=D_CONV, expand=EXPAND,
                         use_fast_path=False, layer_idx=0).to(device)
custom = MambaSSM(d_model=D_MODEL, d_state=D_STATE, d_conv=D_CONV, expand=EXPAND,
                  scan="sequential").to(device)

print(f"official dt_rank={official.dt_rank}  custom dt_rank={custom.dt_rank}")

# ── 1. Shape-assert, then copy every parameter official -> custom ──────────────
pairs = [
    ("in_proj.weight", official.in_proj.weight, custom.in_proj.weight),
    ("conv1d.weight",  official.conv1d.weight,  custom.conv1d.weight),
    ("conv1d.bias",    official.conv1d.bias,    custom.conv1d.bias),
    ("x_proj.weight",  official.x_proj.weight,  custom.x_proj.weight),
    ("dt_proj.weight", official.dt_proj.weight, custom.dt_proj.weight),
    ("dt_proj.bias",   official.dt_proj.bias,   custom.dt_proj.bias),
    ("A_log",          official.A_log,          custom.A_log),
    ("D",              official.D,              custom.D),
    ("out_proj.weight",official.out_proj.weight,custom.out_proj.weight),
]
print(f"\n{'param':16s} {'official shape':>16s} {'custom shape':>16s}")
for name, o, c in pairs:
    assert o.shape == c.shape, f"SHAPE MISMATCH {name}: official {o.shape} vs custom {c.shape}"
    print(f"{name:16s} {str(tuple(o.shape)):>16s} {str(tuple(c.shape)):>16s}  match")
with torch.no_grad():
    for _, o, c in pairs:
        c.data.copy_(o.data)
print("\nall parameters copied official -> custom")

In [ ]:
# Cell 12: Stage-by-stage forward + gradient comparison, on the weight-copied
# models from Cell 11. Both sides walk their OWN submodules manually -- no
# forward(), no opaque fused kernels -- so every number here comes from a plain
# nn.Linear/nn.Conv1d call or the already-parity-tested selective_scan().
#
# The scan step uses our own selective_scan() for BOTH sides (fed each model's
# own dt/B/C/u), since that function is already proven to match the official
# selective_scan_ref (tests/test_selective_scan_parity.py) -- isolating this test
# to everything AROUND the scan: in_proj, conv+SiLU, x_proj, dt_proj, D-skip,
# z-gate, out_proj.
torch.manual_seed(0)
B, T = 4, 32
x = torch.randn(B, T, D_MODEL, device=device)
x_o = x.clone().requires_grad_(True)
x_c = x.clone().requires_grad_(True)

def trace(model, x, tag):
    """Manual walk of one model's own submodules -- mirrors both MambaSSM.forward
    and the official slow-path source (mamba_simple.py) exactly, since they are
    mathematically the same sequence of ops."""
    xz = model.in_proj(x)                                    # (B,T,2E)
    x_in, z = xz.chunk(2, dim=-1)
    h_conv = model.conv1d(x_in.transpose(1, 2))[:, :, :T]     # (B,E,T)
    h_conv = F.silu(h_conv).transpose(1, 2)                   # (B,T,E)
    dBC = model.x_proj(h_conv)                                # (B,T,R+2N)
    dt_rank = model.dt_rank
    d_state = model.A_log.shape[1]
    delta_low = dBC[:, :, :dt_rank]
    B_ssm = dBC[:, :, dt_rank:dt_rank + d_state]
    C_ssm = dBC[:, :, dt_rank + d_state:]
    delta = F.softplus(model.dt_proj(delta_low))              # (B,T,E)
    A = -torch.exp(model.A_log.float())
    y = selective_scan(delta, A, B_ssm, C_ssm, h_conv, scan="sequential")  # (B,T,E)
    y_skip = y + model.D.unsqueeze(0).unsqueeze(0) * h_conv
    y_gated = y_skip * F.silu(z)
    out = model.out_proj(y_gated)
    stages = {"xz": xz, "x_in": x_in, "z": z, "h_conv": h_conv, "delta_low": delta_low,
             "B_ssm": B_ssm, "C_ssm": C_ssm, "delta": delta, "scan_y": y,
             "y_skip": y_skip, "y_gated": y_gated, "out": out}
    return out, stages

out_o, stages_o = trace(official, x_o, "official")
out_c, stages_c = trace(custom, x_c, "custom")

print(f"{'stage':12s} {'max |diff|':>12s}")
print("-" * 26)
for name in stages_o:
    d = (stages_o[name] - stages_c[name]).abs().max().item()
    flag = "" if d < 1e-3 else "  <-- DIVERGES HERE"
    print(f"{name:12s} {d:>12.2e}{flag}")

loss_o = out_o.sum()
loss_c = out_c.sum()
loss_o.backward()
loss_c.backward()

print(f"\ninput grad max |diff|: {(x_o.grad - x_c.grad).abs().max().item():.2e}")
print(f"\n{'param grad':16s} {'max |diff|':>12s}")
print("-" * 30)
for name, o, c in pairs:
    if o.grad is None or c.grad is None:
        print(f"{name:16s}  MISSING (official={o.grad is not None}, custom={c.grad is not None})")
        continue
    d = (o.grad - c.grad).abs().max().item()
    flag = "" if d < 1e-2 else "  <-- DIVERGES HERE"
    print(f"{name:16s} {d:>12.2e}{flag}")

## Training-system diagnostic (only run because Cell 12 parity was exact)

v4 result: forward AND gradient parity on the weight-copied models was **0.00e+00 at
every single stage** -- not approximately equal, bit-identical. `MambaSSM.forward()`
computes exactly what the official implementation computes; the architecture and
computation are fully exonerated. Per the review: stop touching `forward()` entirely.

The remaining search space is construction/training plumbing: initialization (before
any weight-copying), the residual/norm wrapper, optimizer grouping, precision, or a
harness-level detail. This checks that directly on FRESH (uncopied) construction --
same seed, same one minibatch, same one optimizer step -- logging parameter
distributions, post-block activations, gradient norms per parameter, and the actual
parameter delta after one Adam step.

In [ ]:
# Cell 13 (staged, not yet appended): first-optimizer-step diagnostic.
# Only meaningful if Cell 12's parity passed -- if the same weights produce the
# same outputs/gradients, any training gap must come from HOW the two models are
# initialized, wrapped, or optimized, not from what they compute. This checks
# that directly: fresh (uncopied) construction for both, one identical minibatch,
# one optimizer step, compare distributions/moments/deltas before any divergence
# in training data could explain a difference.
import copy

torch.manual_seed(0)
official_fresh = OfficialMamba(d_model=D_MODEL, d_state=D_STATE, d_conv=D_CONV,
                               expand=EXPAND, use_fast_path=False, layer_idx=0).to(device)
torch.manual_seed(0)
custom_fresh = MambaSSM(d_model=D_MODEL, d_state=D_STATE, d_conv=D_CONV,
                        expand=EXPAND, scan="sequential").to(device)

fresh_pairs = [
    ("in_proj.weight", official_fresh.in_proj.weight, custom_fresh.in_proj.weight),
    ("conv1d.weight",  official_fresh.conv1d.weight,  custom_fresh.conv1d.weight),
    ("conv1d.bias",    official_fresh.conv1d.bias,    custom_fresh.conv1d.bias),
    ("x_proj.weight",  official_fresh.x_proj.weight,  custom_fresh.x_proj.weight),
    ("dt_proj.weight", official_fresh.dt_proj.weight, custom_fresh.dt_proj.weight),
    ("dt_proj.bias",   official_fresh.dt_proj.bias,   custom_fresh.dt_proj.bias),
    ("A_log",          official_fresh.A_log,          custom_fresh.A_log),
    ("D",              official_fresh.D,              custom_fresh.D),
    ("out_proj.weight",official_fresh.out_proj.weight,custom_fresh.out_proj.weight),
]

# ── 1. Initial parameter distributions (FRESH construction, no copying) ────────
print(f"{'param':16s} {'official mean/std':>22s} {'custom mean/std':>22s}")
print("-" * 62)
for name, o, c in fresh_pairs:
    om, os_ = o.mean().item(), o.std().item()
    cm, cs = c.mean().item(), c.std().item()
    print(f"{name:16s} {f'{om:+.4f}/{os_:.4f}':>22s} {f'{cm:+.4f}/{cs:.4f}':>22s}")

# ── 2. Activations after one forward pass, same input, fresh (uncopied) weights ─
torch.manual_seed(1)
x0 = torch.randn(B, T, D_MODEL, device=device)
out_of, stages_of = trace(official_fresh, x0.clone().requires_grad_(True), "official_fresh")
out_cf, stages_cf = trace(custom_fresh, x0.clone().requires_grad_(True), "custom_fresh")
print(f"\n{'stage':12s} {'official mean/std':>22s} {'custom mean/std':>22s}")
print("-" * 58)
for name in stages_of:
    om, os_ = stages_of[name].mean().item(), stages_of[name].std().item()
    cm, cs = stages_cf[name].mean().item(), stages_cf[name].std().item()
    print(f"{name:12s} {f'{om:+.4f}/{os_:.4f}':>22s} {f'{cm:+.4f}/{cs:.4f}':>22s}")

# ── 3. Gradient norm per parameter, one backward pass ───────────────────────────
out_of.sum().backward()
out_cf.sum().backward()
print(f"\n{'param':16s} {'official |grad|':>16s} {'custom |grad|':>16s}")
print("-" * 50)
for name, o, c in fresh_pairs:
    og = o.grad.norm().item() if o.grad is not None else float("nan")
    cg = c.grad.norm().item() if c.grad is not None else float("nan")
    print(f"{name:16s} {og:>16.3e} {cg:>16.3e}")

# ── 4. Adam moments + actual parameter delta after ONE optimizer step ──────────
def adam_step_group(model, lr=3e-4, wd=0.1):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        (no_decay if getattr(p, "_no_weight_decay", False) or getattr(p, "_no_reinit", False)
         else decay).append(p)
    opt = torch.optim.AdamW(
        [{"params": decay, "weight_decay": wd}, {"params": no_decay, "weight_decay": 0.0}],
        lr=lr, betas=(0.9, 0.95))
    before = {n: p.detach().clone() for n, p in model.named_parameters()}
    opt.step()
    after = {n: p.detach().clone() for n, p in model.named_parameters()}
    delta = {n: (after[n] - before[n]).abs().mean().item() for n in before}
    return opt, delta

opt_o, delta_o = adam_step_group(official_fresh)
opt_c, delta_c = adam_step_group(custom_fresh)

print(f"\n{'param':30s} {'official |delta|':>16s} {'custom |delta|':>16s}")
print("-" * 64)
names_o = {n for n, _ in official_fresh.named_parameters()}
names_c = {n for n, _ in custom_fresh.named_parameters()}
for n in sorted(names_o | names_c):
    do = delta_o.get(n, float("nan"))
    dc = delta_c.get(n, float("nan"))
    print(f"{n:30s} {do:>16.3e} {dc:>16.3e}")


## Full-stack parity: does composition or checkpointing break something a single mixer doesn't show?

v5 result: bare-mixer parity was exact at every level tested -- initialization,
activations, gradient norms, and the actual parameter delta after one real AdamW step.
That test used ONE mixer, `T=32` (the real task uses `T=64`), and never touched gradient
checkpointing (`MAMBA_GRAD_CHECKPOINT=True` only engages inside `Mamba.forward()`'s
multi-layer stack when `self.training=True`; the isolated `MambaSSM` mixer used in v5
never goes through that path at all).

So two suspects remain untested: multi-layer residual/norm composition, and checkpoint
recomputation. This builds a small (3-layer) official stack (`OracleMamba`, already
defined in Cell 4 -- it mirrors our own `Mamba` class's structure: embedding -> N
residual pre-norm blocks -> final LayerNorm -> tied head) and our own `Mamba`, copies
every weight layer-for-layer (including each block's own LayerNorm, the final `ln_f`,
and the tied embedding), then compares full forward/backward twice: once with our
gradient checkpointing OFF (`custom.eval()` -- checkpointing is gated on
`self.training`), once ON (`custom.train()`, matching what real training actually does
every step). Dropout is 0 in both stacks so there's no other source of stochasticity.

In [ ]:
# Cell 15: Full-stack parity -- 3 layers, weight-copied, checkpointing off vs on.
import torch
import torch.nn as nn
from src.models import build_model

N_STACK_LAYERS = 3
torch.manual_seed(0)
official_stack = OracleMamba(vocab_size=TASK_ARGS["vocab_size"], n_layers=N_STACK_LAYERS,
                             d_model=D_MODEL, d_state=D_STATE, d_conv=D_CONV,
                             expand=EXPAND, use_fast_path=False).to(device)
custom_stack = build_model("mamba", vocab_size=TASK_ARGS["vocab_size"],
                           n_layers=N_STACK_LAYERS, d_model=D_MODEL, d_state=D_STATE,
                           d_conv=D_CONV, expand=EXPAND, dropout=0.0,
                           scan="sequential").to(device)

# custom_stack carries extra ablation-only params (fixed_delta_pre/fixed_B/fixed_C,
# per layer -- see src/models/mamba.py's input_independent path) that the official
# model has no equivalent of at all. They're never used in this (non-ablated)
# comparison, so exclude them from the equivalence check rather than let them
# masquerade as a real architecture mismatch.
ABLATION_ONLY = ("fixed_delta_pre", "fixed_B", "fixed_C")
n_o = sum(p.numel() for p in official_stack.parameters() if p.requires_grad)
n_c = sum(p.numel() for n, p in custom_stack.named_parameters()
         if p.requires_grad and not any(n.endswith(a) for a in ABLATION_ONLY))
n_c_ablation = sum(p.numel() for n, p in custom_stack.named_parameters()
                   if p.requires_grad and any(n.endswith(a) for a in ABLATION_ONLY))
print(f"official_stack: {n_o:,} params   "
      f"custom_stack: {n_c:,} params (+{n_c_ablation:,} ablation-only, excluded)")
assert n_o == n_c, f"param count mismatch: {n_o} vs {n_c} -- structure diverged somewhere"

# ── Copy every weight, layer-for-layer ──────────────────────────────────────────
stack_pairs = []
for i in range(N_STACK_LAYERS):
    ob, cb = official_stack.blocks[i], custom_stack.blocks[i]
    stack_pairs += [
        (f"block{i}.norm.weight", ob.norm.weight, cb.norm.weight),
        (f"block{i}.norm.bias",   ob.norm.bias,   cb.norm.bias),
        (f"block{i}.in_proj.weight",  ob.mixer.in_proj.weight,  cb.ssm.in_proj.weight),
        (f"block{i}.conv1d.weight",   ob.mixer.conv1d.weight,   cb.ssm.conv1d.weight),
        (f"block{i}.conv1d.bias",     ob.mixer.conv1d.bias,     cb.ssm.conv1d.bias),
        (f"block{i}.x_proj.weight",   ob.mixer.x_proj.weight,   cb.ssm.x_proj.weight),
        (f"block{i}.dt_proj.weight",  ob.mixer.dt_proj.weight,  cb.ssm.dt_proj.weight),
        (f"block{i}.dt_proj.bias",    ob.mixer.dt_proj.bias,    cb.ssm.dt_proj.bias),
        (f"block{i}.A_log",           ob.mixer.A_log,           cb.ssm.A_log),
        (f"block{i}.D",               ob.mixer.D,               cb.ssm.D),
        (f"block{i}.out_proj.weight", ob.mixer.out_proj.weight, cb.ssm.out_proj.weight),
    ]
stack_pairs += [
    ("tok_emb.weight", official_stack.tok_emb.weight, custom_stack.tok_emb.weight),
    ("ln_f.weight",    official_stack.ln_f.weight,    custom_stack.ln_f.weight),
    ("ln_f.bias",      official_stack.ln_f.bias,      custom_stack.ln_f.bias),
]

bad = [(n, o.shape, c.shape) for n, o, c in stack_pairs if o.shape != c.shape]
assert not bad, f"shape mismatches: {bad}"
with torch.no_grad():
    for _, o, c in stack_pairs:
        c.data.copy_(o.data)
print(f"copied {len(stack_pairs)} tensors, all shapes matched, official -> custom")

# ── One forward+backward pass per configuration -- official, custom (ckpt off),
# custom (ckpt on). Each call returns BOTH the output and the full gradient dict,
# so this is exactly three passes total, not six. ──────────────────────────────
torch.manual_seed(2)
T_STACK = TASK_ARGS["seq_len"]  # match the real task's T=64, not cell 12's T=32
xb = torch.randint(0, TASK_ARGS["vocab_size"], (4, T_STACK), device=device)

def run_pass(model, x, train_mode):
    # checkpointing is gated on self.training inside Mamba.forward(); eval() ->
    # off, train() -> on (matching what real training actually does every step).
    # LayerNorm is train/eval-invariant and dropout=0 in both stacks, so
    # train_mode ONLY toggles the checkpoint path here, nothing else.
    model.train(train_mode)
    out = model(x.clone())
    loss = out.float().sum()
    model.zero_grad(set_to_none=True)
    loss.backward()
    grads = {n: (p.grad.detach().clone() if p.grad is not None else None)
            for n, p in model.named_parameters()}
    return out.detach(), grads

out_o, grad_o                 = run_pass(official_stack, xb, train_mode=False)
out_c_nockpt, grad_c_nockpt   = run_pass(custom_stack,   xb, train_mode=False)
out_c_ckpt,   grad_c_ckpt     = run_pass(custom_stack,   xb, train_mode=True)

fwd_diff_nockpt = (out_o - out_c_nockpt).abs().max().item()
fwd_diff_ckpt = (out_o - out_c_ckpt).abs().max().item()
fwd_diff_ckpt_vs_nockpt = (out_c_ckpt - out_c_nockpt).abs().max().item()

print(f"\nforward: official vs custom (checkpoint OFF): {fwd_diff_nockpt:.3e}")
print(f"forward: official vs custom (checkpoint ON):  {fwd_diff_ckpt:.3e}")
print(f"forward: custom checkpoint ON vs OFF (self-consistency): {fwd_diff_ckpt_vs_nockpt:.3e}")

print(f"\n{'param':30s} {'off vs official':>16s} {'on vs official':>16s}")
print("-" * 66)
for name, o, c in stack_pairs:
    oname = next(n for n, p in official_stack.named_parameters() if p is o)
    cname = next(n for n, p in custom_stack.named_parameters() if p is c)
    go, gcf, gck = grad_o.get(oname), grad_c_nockpt.get(cname), grad_c_ckpt.get(cname)
    d_off = (go - gcf).abs().max().item() if go is not None and gcf is not None else float("nan")
    d_on  = (go - gck).abs().max().item() if go is not None and gck is not None else float("nan")
    flag = "" if max(d_off, d_on) < 1e-1 else "  <-- DIVERGES"
    print(f"{name:30s} {d_off:>16.3e} {d_on:>16.3e}{flag}")


## Trajectory tracking: when do two identical-at-step-0 stacks first diverge?

If the full-stack parity above passed (both checkpointing off and on), the question stops
being "are these the same computation" and becomes "how many real optimizer steps before
two identical models stop matching." This continues from the already weight-copied,
still-identical `official_stack`/`custom_stack` -- no rebuild -- and runs real AdamW
training (custom in `.train()` mode, checkpointing on, matching production exactly) on
the actual Selective Copying data stream, same seeds `train_task.py` itself uses,
printing the worst per-parameter absolute difference after every step.

In [ ]:
# Cell 17: Track when trajectories first diverge -- IF Cell 16's full-stack parity
# passed, this is the next question per the review: not "are they the same
# computation" (answered) but "how many real optimizer steps before two
# identical-at-step-0 models stop matching."
#
# Continues from Cell 16's already weight-copied, still-identical official_stack /
# custom_stack -- does NOT rebuild them, so their step-0 state is exactly what
# Cell 16 already verified. custom_stack runs in .train() mode (checkpointing ON,
# matching real training exactly -- no need to re-test checkpointing itself here).
import sys, math
sys.path.insert(0, "/kaggle/working/transformer-vs-ssm")
from src.optim import build_param_groups
from src.tasks import IGNORE_INDEX, make_batcher

N_TRACK_STEPS = 30
TRACK_LR = LR  # same 3e-4 as the real training runs
TRACK_WARMUP = WARMUP

def get_lr_local(step, warmup=TRACK_WARMUP, max_steps=STEPS, lr=TRACK_LR):
    if step < warmup:
        return lr * (step + 1) / warmup
    prog = (step - warmup) / max(1, max_steps - warmup)
    return lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))

# Official: same decay/no-decay-by-attribute convention used in Cell 8's oracle training.
o_decay, o_no_decay = [], []
for n, p in official_stack.named_parameters():
    (o_no_decay if getattr(p, "_no_weight_decay", False) else o_decay).append(p)
opt_o = torch.optim.AdamW(
    [{"params": o_decay, "weight_decay": 0.1}, {"params": o_no_decay, "weight_decay": 0.0}],
    lr=TRACK_LR, betas=(0.9, 0.95))

# Custom: the ACTUAL src/optim.py grouping used by real training (train_task.py),
# ssm_lr_scale=1.0 default -- not a hand-rolled approximation of it.
opt_c = torch.optim.AdamW(
    build_param_groups(custom_stack, weight_decay=0.1, ssm_lr_scale=1.0),
    lr=TRACK_LR, betas=(0.9, 0.95))

official_stack.train()
custom_stack.train()  # checkpointing ON -- matches real training exactly

batcher = make_batcher("selective_copy", **TASK_ARGS)
print(f"{'step':>5} {'lr':>9}  {'max |param diff| across whole stack':>36}  {'worst param':>20}")
print("-" * 78)

for step in range(N_TRACK_STEPS):
    lr = get_lr_local(step)
    for g in opt_o.param_groups:
        g["lr"] = lr
    for g in opt_c.param_groups:
        g["lr"] = lr * g.get("lr_scale", 1.0)

    # SAME minibatch fed to both models -- matches train_task.py's own seeding
    # convention (seed = step + seed*10_000_000) so this is the same data stream
    # real training would see, not fresh random noise disconnected from the task.
    tok, tgt, _, _ = batcher(BATCH, seed=step)
    tok = torch.from_numpy(tok).to(device)
    tgt = torch.from_numpy(tgt).to(device)

    logits_o = official_stack(tok)
    loss_o = torch.nn.functional.cross_entropy(
        logits_o.reshape(-1, TASK_ARGS["vocab_size"]), tgt.reshape(-1),
        ignore_index=IGNORE_INDEX)
    opt_o.zero_grad(set_to_none=True)
    loss_o.backward()
    opt_o.step()

    logits_c = custom_stack(tok)
    loss_c = torch.nn.functional.cross_entropy(
        logits_c.reshape(-1, TASK_ARGS["vocab_size"]), tgt.reshape(-1),
        ignore_index=IGNORE_INDEX)
    opt_c.zero_grad(set_to_none=True)
    loss_c.backward()
    opt_c.step()

    # Compare every parameter, by NAME within each model (not shared object
    # identity -- these have been independent tensors since Cell 16's copy).
    worst_diff, worst_name = 0.0, None
    for name, o, c in stack_pairs:
        oname = next(n for n, p in official_stack.named_parameters() if p is o)
        cname = next(n for n, p in custom_stack.named_parameters() if p is c)
        d = (o - c).abs().max().item()
        if d > worst_diff:
            worst_diff, worst_name = d, name

    loss_diff = abs(loss_o.item() - loss_c.item())
    print(f"{step+1:>5} {lr:>9.2e}  {worst_diff:>36.3e}  {worst_name:>20}  "
          f"(loss_o={loss_o.item():.4f} loss_c={loss_c.item():.4f} diff={loss_diff:.2e})")

print("\nIf worst |param diff| stays near float32 noise (~1e-6 to 1e-5) throughout, the")
print("two stacks are tracking together and the training-time gap is NOT visible in")
print("early steps -- it would have to emerge from something that compounds over many")
print("more steps (numerical drift in the 64-step sequential scan, or a step-budget/")
print("annealing effect). If it grows past ~1e-2 within these 30 steps, THAT step")
print("number and worst_name localize exactly where the two training paths split.")
